In [11]:
import os
import pymysql

# 连接MySQL数据库
connection = pymysql.connect(
    host='localhost',
    user='root',
    password=os.environ.get('MYSQL_PASSWORD', ''),
    database='hms'
)

try:
    with connection.cursor() as cursor:
        # 查询某月份某星期的数据
        query = '''
            SELECT 
                YEAR(orderdate) AS year,
                MONTH(orderdate) AS month,
                WEEK(orderdate) AS week,
                COUNT(*) AS count
            FROM 
                orders
            WHERE 
                YEAR(orderdate) = 2023 AND MONTH(orderdate) = 11
            GROUP BY 
                YEAR(orderdate), MONTH(orderdate), WEEK(orderdate)
            ORDER BY 
                YEAR(orderdate), MONTH(orderdate), WEEK(orderdate)
        '''

        # 执行查询
        cursor.execute(query)

        # 获取查询结果
        result = cursor.fetchall()

        # 构建结果列表
        data = []
        for row in result:
            year = row[0]
            month = row[1]
            week = row[2]
            count = row[3]
            data.append({
                'year': year,
                'month': month,
                'week': week,
                'count': count
            })

        # 打印结果
        for item in data:
            print(item)

finally:
    # 关闭数据库连接
    connection.close()

{'year': 2023, 'month': 11, 'week': 48, 'count': 5}


In [10]:
import os
import pymysql
import datetime

# 连接MySQL数据库
conn = pymysql.connect(host='localhost', 
                       user='root', 
                       password=os.environ.get('MYSQL_PASSWORD', ''),
                       database='hms',
                       charset='utf8mb4')
                           
# 通过Cursor执行SQL语句        
cursor = conn.cursor()

# 指定统计月份
month = 11 

# 统计每个星期几的订单数量
sql = "SELECT DAYOFWEEK(orderdate) day, COUNT(*) count FROM orders WHERE MONTH(orderdate)={} GROUP BY day".format(month)

cursor.execute(sql)

# 获取结果集
results = cursor.fetchall()

# 打印结果 
for result in results:
    day = ''
    if result[0] == 1:
        day = '星期一'
    elif result[0] == 2:
        day = '星期二'
    elif result[0] == 3:
        day = '星期三' 
    elif result[0] == 4:
        day = '星期四'
    elif result[0] == 5:
        day = '星期五'
    elif result[0] == 6:
        day = '星期六'
    elif result[0] == 7:  
        day = '星期天'
        
    print('{}有{}条数据'.format(day, result[1]))
        
# 关闭连接
cursor.close()
conn.close()

星期二有2条数据
星期三有1条数据
星期四有1条数据
星期五有1条数据


In [10]:
import os
import pymysql
import csv

conn = pymysql.connect(host='localhost', 
                       user='root', 
                       password=os.environ.get('MYSQL_PASSWORD', ''),
                       database='hms')
                       
cursor = conn.cursor()

# 构建SQL查询所有月份和星期的订单条数       
sql = """
SELECT MONTH(orderdate) month, DAYOFWEEK(orderdate) day, COUNT(*) count  
FROM orders
GROUP BY month, day
"""

cursor.execute(sql)
results = cursor.fetchall()

# 将结果写入CSV文件 
fieldnames = ['month', 'day', 'count']
with open('train_data.csv', 'w') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows([{'month':month, 'day':day, 'count':count} for month,day,count in results])
    
print(results)
for i in results:
    print(i)
cursor.close()
conn.close()

((11, 2, 2), (11, 3, 1), (11, 4, 1), (11, 5, 1), (12, 2, 1), (12, 1, 1), (12, 7, 1), (12, 6, 1))
(11, 2, 2)
(11, 3, 1)
(11, 4, 1)
(11, 5, 1)
(12, 2, 1)
(12, 1, 1)
(12, 7, 1)
(12, 6, 1)


In [1]:
import os
import pymysql
import pandas as pd

# 连接MySQL
connection = pymysql.connect(host='localhost',
                             user='root',
                             password=os.environ.get('MYSQL_PASSWORD', ''),
                             database='hms')

# 使用Cursor执行SQL语句        
cursor = connection.cursor()

# 查询所有月份、星期和订单数量
sql = """
SELECT 
  MONTH(orderdate) month,
  DAYOFWEEK(orderdate) day, 
  COUNT(*) count
FROM orders  
GROUP BY month, day
"""

# 执行查询
cursor.execute(sql)

# 获取结果并生成DataFrame
results = cursor.fetchall()
df = pd.DataFrame(results, columns=['month', 'day', 'count']) 

# 调整星期数字段,使星期一为1
df['day'] = df['day'].map({1:7, 2:1, 3:2, 4:3, 5:4, 6:5, 7:6})

# 保存为CSV文件
df.to_csv('orders_data.csv', index=False)

# 关闭连接
cursor.close()
connection.close()

In [20]:
import torch
import torch.nn as nn
import pandas as pd
from torch.utils.data import DataLoader, Dataset

# 导入数据
data = pd.read_csv('orders_data.csv')

# 分离输入输出
X = data[['month', 'day']] 
y = data['count']
# X为DataFrame,转换为Tensor
X = torch.tensor(X.values, dtype=torch.float32)
y = torch.tensor(y.values, dtype=torch.float32)


# 自定义Dataset类 
class OrderDataset(Dataset):
  def __init__(self, X, y):
    self.X = X
    self.y = y

  def __len__(self):
    return len(self.X)

  def __getitem__(self, index):
    return self.X[index], self.y[index]

# 将数据封装进Dataset
dataset = OrderDataset(X, y) 

# 使用DataLoader加载小批量数据
batch_size = 5
dataloader = DataLoader(dataset, batch_size, shuffle=True)

# 定义模型
# 定义神经网络模型
class Net(nn.Module):
  def __init__(self):
    super().__init__()
    self.fc1 = nn.Linear(2, 10) 
    self.fc2 = nn.Linear(10, 5)
    self.fc3 = nn.Linear(5, 1)

  def forward(self, x):
    x = torch.tanh(self.fc1(x))
    x = torch.tanh(self.fc2(x))
    x = self.fc3(x)
    return x

model = Net()

# 优化器和损失函数
optimizer = torch.optim.SGD(model.parameters(), lr=0.001)
loss_func = nn.MSELoss()



# 训练循环
for epoch in range(10):
      for xb, yb in dataloader:

        # 前向和反向传播
        pred = model(xb)
        loss = loss_func(pred, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
      if epoch%1 == 0:
        print(epoch, loss.item())

# 不需要反向传播,所以使用no_grad
with torch.no_grad():

    # 在训练数据集上测试
    y_pred = model(X_train)

    # 计算误差
    loss = loss_func(y_pred, y_train)
    print('Train loss:', loss.item())

0 0.002156503964215517
1 0.024896064773201942
2 0.311559796333313
3 0.038316965103149414
4 0.016836367547512054
5 0.02382420189678669
6 0.011411046609282494
7 0.03789354860782623
8 0.029090575873851776
9 0.015609653666615486


C:\anaconda\lib\site-packages\torch\nn\modules\loss.py:536: UserWarning: Using a target size (torch.Size([5])) that is different to the input size (torch.Size([5, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
C:\anaconda\lib\site-packages\torch\nn\modules\loss.py:536: UserWarning: Using a target size (torch.Size([3])) that is different to the input size (torch.Size([3, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


NameError: name 'X_train' is not defined

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd

# 定义数据集类
class MyDataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        return self.data[index]

# 从 CSV 文件中加载数据
data = pd.read_csv("orders_data.csv")

# 数据预处理
processed_data = []
for index, row in data.iterrows():
    month = row['month']
    day = row['day']
    count = row['count']
    # 对月份和星期几进行编码为 one-hot 向量
    month_one_hot = [int(month == i) for i in range(1, 13)]
    day_one_hot = [int(day == i) for i in range(1, 8)]
    processed_data.append(month_one_hot + day_one_hot + [count])

# 将处理后的数据转换为张量
tensor_data = torch.tensor(processed_data, dtype=torch.float)

# 将数据拆分为训练集和验证集
train_size = int(0.8 * len(tensor_data))
val_size = len(tensor_data) - train_size
train_data, val_data = torch.utils.data.random_split(tensor_data, [train_size, val_size])

# 定义模型
class MyModel(nn.Module):
    def __init__(self):
        super(MyModel, self).__init__()
        self.fc1 = nn.Linear(19, 16)
        self.fc2 = nn.Linear(16, 8)
        self.fc3 = nn.Linear(8, 1)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)
        return x

# 创建模型，并定义损失函数和优化器
model = MyModel()
loss_fn = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 创建用于训练和验证的数据加载器
batch_size = 2
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_data, batch_size=batch_size)

# 训练模型
num_epochs = 100
for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0
    for inputs in train_loader:
        optimizer.zero_grad()
        outputs = model(inputs[:, :-1])
        targets = inputs[:, -1:]
        loss = loss_fn(outputs, targets)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * inputs.size(0)
    train_loss /= len(train_loader.dataset)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for inputs in val_loader:
            outputs = model(inputs[:, :-1])
            targets = inputs[:, -1:]
            loss = loss_fn(outputs, targets)
            val_loss += loss.item() * inputs.size(0)
        val_loss /= len(val_loader.dataset)

    print(f"Epoch {epoch+1}/{num_epochs}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")

# 保存训练好的模型
torch.save(model.state_dict(), 'model.pth')

Epoch 1/100, Train Loss: 11.2404, Val Loss: 4.7197
Epoch 2/100, Train Loss: 11.1337, Val Loss: 4.6489
Epoch 3/100, Train Loss: 11.0218, Val Loss: 4.5755
Epoch 4/100, Train Loss: 10.8980, Val Loss: 4.4954
Epoch 5/100, Train Loss: 10.7658, Val Loss: 4.4091
Epoch 6/100, Train Loss: 10.6147, Val Loss: 4.3151
Epoch 7/100, Train Loss: 10.4535, Val Loss: 4.2132
Epoch 8/100, Train Loss: 10.2700, Val Loss: 4.1011
Epoch 9/100, Train Loss: 10.0797, Val Loss: 3.9803
Epoch 10/100, Train Loss: 9.8652, Val Loss: 3.8499
Epoch 11/100, Train Loss: 9.6428, Val Loss: 3.7129
Epoch 12/100, Train Loss: 9.3929, Val Loss: 3.5638
Epoch 13/100, Train Loss: 9.1239, Val Loss: 3.4051
Epoch 14/100, Train Loss: 8.8565, Val Loss: 3.2347
Epoch 15/100, Train Loss: 8.5404, Val Loss: 3.0505
Epoch 16/100, Train Loss: 8.2141, Val Loss: 2.8581
Epoch 17/100, Train Loss: 7.8373, Val Loss: 2.6529
Epoch 18/100, Train Loss: 7.4624, Val Loss: 2.4340
Epoch 19/100, Train Loss: 7.0458, Val Loss: 2.2027
Epoch 20/100, Train Loss: 6.588

In [31]:
model(torch.tensor([[int(12 == i) for i in range(1, 13)],\
                    [int(2 == i) for i in range(1, 8)]],\
                   dtype=torch.float32))

ValueError: expected sequence of length 12 at dim 1 (got 7)

In [30]:
model(torch.tensor([12,2], dtype=torch.float32))

RuntimeError: mat1 and mat2 shapes cannot be multiplied (1x2 and 19x16)

In [1]:
import torch
import torch.nn as nn

# 定义模型
class MyModel(nn.Module):
    def __init__(self):
        super(MyModel, self).__init__()
        self.fc1 = nn.Linear(19, 16)
        self.fc2 = nn.Linear(16, 8)
        self.fc3 = nn.Linear(8, 1)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)
        return x

# 创建模型实例
model = MyModel()

# 加载训练好的模型参数
model.load_state_dict(torch.load('model.pth'))

<All keys matched successfully>

In [2]:
import torch

# 定义输入值
input_month = 11
input_day = 1

# 对输入值进行编码为 one-hot 向量
input_month_one_hot = [int(input_month == i) for i in range(1, 13)]
input_day_one_hot = [int(input_day == i) for i in range(1, 8)]

# 将输入值转换为张量
input_tensor = torch.tensor(input_month_one_hot + input_day_one_hot, dtype=torch.float).unsqueeze(0)

In [3]:
# 将模型设为评估模式
model.eval()

# 使用模型进行推断
with torch.no_grad():
    output = model(input_tensor)

# 输出预测结果
predicted_count = output.item()
print(f"预测的订单数量为：{predicted_count}")

预测的订单数量为：1.5370068550109863


In [7]:
round(output.item())

2

In [ ]:
import socket
import torch
import torch.nn as nn
import datetime
import json


# 定义模型
class MyModel(nn.Module):
    def __init__(self):
        super(MyModel, self).__init__()
        self.fc1 = nn.Linear(19, 16)
        self.fc2 = nn.Linear(16, 8)
        self.fc3 = nn.Linear(8, 1)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)
        return x

# 创建模型实例
model = MyModel()

# 加载训练好的模型参数
model.load_state_dict(torch.load('model.pth'))

def m(mo,we):
    # 定义输入值
    input_month = mo
    input_day = we

    # 对输入值进行编码为 one-hot 向量
    input_month_one_hot = [int(input_month == i) for i in range(1, 13)]
    input_day_one_hot = [int(input_day == i) for i in range(1, 8)]

    # 将输入值转换为张量
    input_tensor = torch.tensor(input_month_one_hot + input_day_one_hot, dtype=torch.float).unsqueeze(0)
    
    # 将模型设为评估模式
    model.eval()

    # 使用模型进行推断
    with torch.no_grad():
        output = model(input_tensor)

    # 输出预测结果
    return output.item()
#     print(f"预测的订单数量为：{predicted_count}")
    
# 服务器地址和端口
HOST = 'localhost'
PORT = 50000

# 创建TCP套接字
server_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)

# 绑定地址和端口
server_socket.bind((HOST, PORT))

# 监听连接
server_socket.listen(1)
print('等待客户端连接...')

while True:
    # 接受连接
    client_socket, client_address = server_socket.accept()
    print('客户端已连接:', client_address)


    # 接收数据
    data = client_socket.recv(1024)
    if not data:
        break
    print('收到客户端消息:', data.decode())
    # 获取今天的日期
    out=[]
    today = datetime.date.today()
    for i in range(1, 8):
        future_date = today + datetime.timedelta(days=i)
        weekday = future_date.weekday() + 1 if future_date.weekday() < 6 else 7
        out.append(m(future_date.month, weekday))
        
    # 创建一个包含键名为 "data" 的字典
    data_dict = {"data": out}

    # 将字典转换为 JSON 字符串
    json_data = json.dumps(data_dict)
    print(json_data)
    # 发送响应
    client_socket.sendall(json_data.encode())

    # 关闭连接
    client_socket.close()
server_socket.close()

等待客户端连接...
客户端已连接: ('127.0.0.1', 58965)
收到客户端消息: Hello, Server!

{"data": [2.2339553833007812, 2.7646842002868652, 2.874300003051758, 2.75302791595459, 2.686262607574463, 2.7899627685546875, 2.2386679649353027]}
客户端已连接: ('127.0.0.1', 59112)
收到客户端消息: Hello, Server!

{"data": [2.2339553833007812, 2.7646842002868652, 2.874300003051758, 2.75302791595459, 2.686262607574463, 2.7899627685546875, 2.2386679649353027]}
客户端已连接: ('127.0.0.1', 60268)
收到客户端消息: Hello, Server!

{"data": [2.2339553833007812, 2.7646842002868652, 2.874300003051758, 2.75302791595459, 2.686262607574463, 2.7899627685546875, 2.2386679649353027]}
客户端已连接: ('127.0.0.1', 60499)
收到客户端消息: Hello, Server!

{"data": [2.2339553833007812, 2.7646842002868652, 2.874300003051758, 2.75302791595459, 2.686262607574463, 2.7899627685546875, 2.2386679649353027]}


In [ ]:
import json

numbers = [1, 2, 3, 4, 5, 6, 7]

# 创建一个包含键名为 "data" 的字典
data_dict = {"data": numbers}

# 将字典转换为 JSON 字符串
json_data = json.dumps(data_dict)

print(json_data)

In [8]:
import datetime

# 获取今天的日期
today = datetime.date.today()

# 输出今天的月份和星期几
print("今天是：{}月{}日 星期{}".format(today.month, today.day, today.strftime("%A")))

# 输出未来七天的日期和星期几（以星期一为第一天）
for i in range(1, 8):
    future_date = today + datetime.timedelta(days=i)
    weekday = future_date.weekday() + 1 if future_date.weekday() < 6 else 7
    print(future_date.month, weekday)

今天是：12月12日 星期Tuesday
12 3
12 4
12 5
12 6
12 7
12 1
12 2


In [3]:
today

datetime.date(2023, 12, 12)

In [6]:
for i in range(1, 8):
    future_date = today + datetime.timedelta(days=i)
    print("未来第{}天是：{}月{}日 星期{}".format(i, future_date.month, future_date.day, future_date.strftime()))

TypeError: strftime() missing required argument 'format' (pos 1)